# 📡 ECMRadar
**Radar sulla formazione medica italiana** — esegui le celle in ordine.

| Step | Cosa fa | Tempo |
|---|---|---|
| **0. Setup** | Monta Drive, clona repo, installa deps | 1 min |
| **1. Calibrazione** | Ispeziona campi ASP.NET reali | 30 sec |
| **2. Test** | Ricerca singola per verificare parsing | 2 min |
| **3. Dump** | Scraping lista eventi per trimestre | 1-2 ore/anno |
| **4. Enrichment** | Speaker/sponsor a blocchi da 500 | ~2s/evento |
| **5. Export** | CSV + Excel su Drive | 30 sec |
| **6. Analisi** | Profiling + Pandas | ∞ |

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ⚠️ MODIFICA con il tuo URL GitHub
REPO_URL = 'https://github.com/TUO_USER/ecmradar.git'

!git clone {REPO_URL} /content/ecmradar 2>/dev/null || (cd /content/ecmradar && git pull)
%cd /content/ecmradar
!pip install -r requirements.txt -q

In [ ]:
import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/ECMRadar')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = str(DRIVE_DIR / 'ecm_database.db')

print(f'DB: {DB_PATH}')
print(f'Drive: {DRIVE_DIR}')

---
## 1. Calibrazione campi ASP.NET
**Solo la prima volta.** Mostra i nomi reali dei campi del form.

In [ ]:
!python inspect_fields.py --save

In [ ]:
# Leggi il mapping generato
import json
try:
    with open('field_mapping.json') as f:
        fields = json.load(f)
    print('Campi trovati:')
    for name, info in fields.get('fields', {}).items():
        print(f"  {name:55s} type={info['type']}")
except FileNotFoundError:
    print('⚠️ Esegui prima la cella sopra')

⚠️ **Se i nomi campo sono diversi** da quelli in `scraper.py → build_search_form()`, aggiornali.
Puoi editare il file direttamente dal pannello sinistro di Colab.

---
## 2. Test — ricerca singola

In [ ]:
# Test: un trimestre, una regione
!python scraper.py \
    --region Lombardia \
    --date-from 01/01/2025 --date-to 31/03/2025 \
    --db {DB_PATH}

In [ ]:
# Verifica DB
import sqlite3
conn = sqlite3.connect(DB_PATH)
for t in ['events', 'providers', 'sponsors', 'speakers']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:15s} {n:>6,} righe')
conn.close()

---
## 3. Dump massivo
Scarica la lista eventi (senza dettagli = veloce).

In [ ]:
# Anno 2025
!python batch_scrape.py --year 2025 --db {DB_PATH}

In [ ]:
# Backfill multi-anno (opzionale)
# !python batch_scrape.py --backfill 2022 2025 --db {DB_PATH}

---
## 4. Enrichment incrementale
Arricchisce **solo** gli eventi senza speaker/sponsor. Puoi interrompere e riprendere.

In [ ]:
# Stato attuale
!python enrich.py --db {DB_PATH} --stats

In [ ]:
# Arricchisci 500 eventi (~17 min)
!python enrich.py --db {DB_PATH} --limit 500

In [ ]:
# Controlla progresso
!python enrich.py --db {DB_PATH} --stats

---
## 5. Export su Drive

In [ ]:
!python export.py --db {DB_PATH} --format csv --output {DRIVE_DIR}/ecm_csv
!python export.py --db {DB_PATH} --format xlsx --output {DRIVE_DIR}/ecm_export.xlsx

print('\n📁 File su Drive:')
!ls -lh {DRIVE_DIR}/

---
## 6. Analisi — Profiler

In [ ]:
import sys
sys.path.insert(0, '.')
from profiler import ECMProfiler

p = ECMProfiler(DB_PATH)

In [ ]:
# Top Provider
p.print_table(p.top_providers(20), 'TOP PROVIDER')

In [ ]:
# Top Sponsor
p.print_table(p.top_sponsors(20), 'TOP SPONSOR')

In [ ]:
# Top KOL
p.print_table(p.top_speakers(20), 'TOP KOL')

In [ ]:
# KOL per area terapeutica
AREA = 'gastroenterologia'  # cambia qui
p.print_table(p.kol_mapping(AREA), f'KOL — {AREA.upper()}')

In [ ]:
# Speaker ↔ Sponsor
p.print_table(p.speaker_sponsor_links(30), 'SPEAKER ↔ SPONSOR')

In [ ]:
# Network di un KOL specifico
import json
SPEAKER = 'Rossi'  # cambia qui
net = p.speaker_network(SPEAKER)
print(json.dumps(net, indent=2, ensure_ascii=False, default=str))

In [ ]:
# Footprint aziendale
AZIENDA = 'Alfasigma'  # cambia qui
fp = p.pharma_footprint(AZIENDA)
print(f'Come sponsor:  {fp["total_sponsored"]} eventi')
print(f'Come provider: {fp["total_organized"]} eventi')

In [ ]:
# Matrice Sponsor × Provider
p.print_table(p.sponsor_provider_matrix(), 'MATRICE SPONSOR × PROVIDER')

---
## 7. Analisi libera con Pandas

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(DB_PATH)

df_events = pd.read_sql('SELECT * FROM events', conn)
df_speakers = pd.read_sql('SELECT * FROM speakers', conn)
df_sponsors = pd.read_sql('SELECT * FROM sponsors', conn)

# Vista flat
df_flat = pd.read_sql('''
    SELECT e.*, p.name as provider_name,
           sp.speaker_name, sp.role, sp.qualifica,
           s.sponsor_name
    FROM events e
    LEFT JOIN providers p ON e.provider_id = p.provider_id
    LEFT JOIN speakers sp ON e.event_id = sp.event_id
    LEFT JOIN sponsors s ON e.event_id = s.event_id
''', conn)
conn.close()

print(f'Events:   {len(df_events):>8,}')
print(f'Speakers: {len(df_speakers):>8,}')
print(f'Sponsors: {len(df_sponsors):>8,}')
print(f'Flat:     {len(df_flat):>8,}')

In [ ]:
# Top 10 sponsor per volume
df_flat.groupby('sponsor_name').agg(
    n_events=('event_id', 'nunique'),
    n_speakers=('speaker_name', 'nunique'),
    n_regions=('region', 'nunique')
).sort_values('n_events', ascending=False).head(10)

In [ ]:
# Pivot speaker × sponsor (top 15 × top 10)
pivot = df_flat.dropna(subset=['speaker_name', 'sponsor_name']).groupby(
    ['speaker_name', 'sponsor_name']
)['event_id'].nunique().reset_index(name='n')

pt = pivot.pivot_table(index='speaker_name', columns='sponsor_name', values='n', fill_value=0)
top_sp = pivot.groupby('speaker_name')['n'].sum().nlargest(15).index
top_ss = pivot.groupby('sponsor_name')['n'].sum().nlargest(10).index
pt.loc[top_sp, top_ss]

In [ ]:
p.close()